# Update & import Python modules

In [ ]:
# install and download required modules
!pip install --upgrade spacy
!python -m spacy download en_core_web_lg
!pip install neo4j
!pip install py2neo

# spaCy
import spacy
from spacy.matcher import Matcher
from spacy.matcher import DependencyMatcher
from spacy.matcher import PhraseMatcher
from spacy.tokens import Span
from spacy import displacy
import numpy as np
from tqdm import tqdm

# Google Drive
from google.colab import drive

# Firebase/Firestore
import firebase_admin
from firebase_admin import credentials
from firebase_admin import firestore

# Neo4j
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable

#py2neo
from py2neo import Node, Graph, Relationship, NodeMatcher
from py2neo.bulk import merge_nodes

# general Python modules
import json
import logging
import re
import requests
from pprint import pprint

In [ ]:
###########################################
# Google Drive
###########################################

from google.colab import drive
drive.mount('/content/drive')
# open Neo4j credentials
with open("/content/drive/MyDrive/ie_course_2022_team08/credentials/neo4j_credentials.json") as f:
  neo4j_credential = json.load(f)

Mounted at /content/drive


In [ ]:
relations_dicts = [
  {
    "pid": "P1327",
    "label": "partners",
    "description": "professional collaboration between companies",
    "aliases": [
        "partner in business or sport",
        "joined forces",
        "partners with",
        "partnered with",
        "collaborates with",
        "partnership",
        "partnered"
    ]
},
{
    "pid": "P899277",
    "label": "recruitment",
    "description": "process of attracting, selecting and appointing candidates to a job or other organization",
    "aliases": [
        "hiring",
        "seeking",
        "vacancy"
    ]
},
{
    "pid": "P1951",
    "label": "investor",
    "description": "individual or organization which invests money in the item for the purpose of obtaining financial return on their investment",
    "aliases": [
        "invests",
        "investing",
        "invested",
        "investment",
        "invested in"
    ]
},
{
    "pid": "P61", # https: //www.wikidata.org/wiki/Property:P61
    "label": "discoverer or inventor",
    "description": "subject who discovered, first described, invented, or developed this discovery or invention",
    "aliases": [
        "coined",
        "inventor",
        "developer",
        "created by",
        "developed by",
        "discoverer",
        "invented",
        "devised by",
        "discovered by",
        "first described",
        "introduced by",
        "invented by",
        "inventor or discoverer",
        "discovered" # added manually
    ]
},
{
    "pid": "P144", # https: //www.wikidata.org/wiki/Property:P3975
    "label": "based on",
    "description": "the work(s) used as the basis for subject item",
    "aliases": [
        "adaptation of",
        "cover of",
        "derivative of"
    ]
}
]

entities_dicts = [
{
    "qid": "Q328840",
    "label": "Visa",
    "description": "American multinational financial services corporation",
    "aliases": [],
    "ner_label": "COMPANY"
},
{
    "qid": "Q16197959",
    "label": "Vitalik Buterin",
    "description": "Russian computer scientist",
    "aliases": [],
    "ner_label": "PERSON"
},
{
    "qid": "Q47273538",
    "label": "Cardano",
    "description": "decentralized, open-source blockchain network that can execute peer-to-peer transactions and serve as a platform for deploying smart contracts",
    "aliases": [],
    "ner_label": "Cryptocurrency"
},
{
    "qid": "Q57665", # https: //www.wikidata.org/wiki/Q57665
    "label": "Jens Stoltenberg",
    "description": "34th Prime Minister of Norway, 13th Secretary-General of NATO",
    "ner_label": "PERSON"
},
{
    "qid": "Q478214",
    "label": "Tesla",
    "description": "American automotive, energy storage and solar power company",
    "aliases": [],
    "ner_label": ""
},
{
    "qid": "Q5835668",
    "label": "Santander Bank",
    "description": "wholly owned subsidiary of Spanish Santander Group",
    "aliases": [],
    "ner_label": "BANK"
},
{
    "qid": "Q22687",
    "label": "bank",
    "description": "financial institution that accepts deposits",
    "aliases": [
        "banks"
    ],
    "ner_label": "COMPANY"
},
{
    "qid": "Q1307473",
    "label": "Ripple",
    "description": "Payment protocol",
    "aliases": [],
    "ner_label": "COMPANY"
},
{
    "qid": "Q29583124",
    "label": "Ethereum Classic",
    "description": "open source blockchain computing platform developed by Charlie Shrem",
    "aliases": [
        "ETC",
        "ETH",
        "Etherium",
        "Ethereum"
    ],
    "ner_label": "CRYPTOCURRENCY",
},
{
    "qid": "Q485178",
    "label": "analyst",
    "description": "profession",
    "aliases": [
        "Analytiker",
        "VC analyst",
        "Software analyst",
        "software analysts"
    ],
    "ner_label": "PROFESSION",
},
{
    "qid": "Q40186999",
    "label": "cryptocurrency wallet",
    "description": "medium like Polygon to store public and/or private keys for encrypting and/or signing information",
    "aliases": [
        "Polygon",
        "cryptocurrency platforms",
        "cryptocurrency",
        "Stellar"
    ],
    "ner_label": "CRYPTOCURRENCY",
},
{
    "qid": "Q20514253",
    "name": "Blockchain",
    "label": "",
    "description": "distributed data store for digital transactions",
    "aliases": [
        "block chain",
        "block chain technology",
        "blockchains",
        "BTC",
        "blockchain technology",
        "Bitcoin",
        "bitcoin"
    ]
},
{
    "qid": "Q41273538",
    "label": "IBM",
    "description": "Multinational company",
    "aliases": [],
    "ner_label": "COMPANY"
},
{
    "qid": "Q15711672",
    "label": "Charlie Shrem",
    "description": "Ethereum Classic is developed by Charlie Shrem",
    "ner_label": "PERSON",
},
{
    "qid": "Q47488338",
    "label": "KodakCoin",
    "description": "discontinued cryptocurrency for photography copyrights",
    "ner_label": "PRODUCT",
}
]

In [ ]:
# dictionary of relations
relations_dictionary = [
    "based on",
    "invested in",
    "secretary general",
    "general secretary",
    "perpetual secretary",
    "leader",
    "vacancy",
    "partnership",
    "coined",
    "inventor",
    "partner in business or sport",
    "partners",
    "partners with",
    "partnered with",
    "partnered",
    "collaborates with",
    "developer",
    "investment",
    "seeking",
    "created by",
    "developed by",
    "discoverer",
    "invented",
    "hiring",
    "recruitment",
    "devised by",
    "discovered by",
    "first described",
    "introduced by",
    "invented by",
    "inventor or discoverer",
    "discovered",
    "joined forces",
    "invests",
    "investor",
    "investing",
    "invested"
]

# dictionary of entities
entities_dictionary = [
    "Vitalik Buterin",
    "Santander Bank",
    "Charlie Shrem",
    "Blockchain",
    "Stellar",
    "BTC",
    "bank",
    "IBM",
    "Tesla",
    "banks",
    "Jens Stoltenberg",
    "Polygon",
    "Ripple",
    "ETH",
    "Cardano",
    "Ethereum",
    "analyst",
    "VC analyst",
    "Visa",
    "cryptocurrency platforms",
    "Software analyst",
    "cryptocurrency wallet",
    "Ethereum Classic",
    "software analysts",
    "cryptocurrency",
    "Bitcoin",
    "bitcoin"
]

sentences = [
    "Ethereum was developed by Vitalik Buterin in 2014",
    "Santander Bank is exploring the possibility of investing in Polygon to expand its payment system capabilities.",
    "Polygon formed a strategic partnership with Santander Bank to create an innovative cross-border payment platform.",
    "Ethereum Classic was developed by Charlie Shrem in his garage", #Passive sentence
    "Polygon is hiring VC analyst",
    "Ethereum Classic is hiring Software analyst",
    "Charlie Shrem discovered Ethereum Classic",
    "Charlie Shrem discovered Ethereum Classic",
    "Etherium is based on BTC for some of its components was the statement made by its founder in 2022",
    "Ripple partnered with Visa to improve their payment system.",
    "Ripple partners with banks to improve cross-border payments",
    "Visa partners with several cryptocurrency platforms to enable the use of cryptocurrencies for everyday purchases",
    "Blockchain and BTC is based on cryptocurrency.",
    "Ripple partnered with Santander Bank to create a cross-border payments platform.",
    "Ripple was partnered with Polygon to improve their payment system.",
    "Charlie Shrem is seeking software analysts for Ethereum Classic development.",
    "Etherium has a vacancy for a VC analyst position.",
    "Visa collaborates with cryptocurrency platforms to facilitate the everyday use of digital currencies.",
    "The founder of Ethereum stated in 2023 that some of its components would be based on Ripple.",
    "IBM invested in Stellar and caused a surge in interest and investment in the cryptocurrency market.",
    "Santander Bank joined forces with Visa to develop a cross-border payments platform.",
    "Tesla invests $1.5 billion in Bitcoin.",
    "Visa is considering investing in Ripple to further enhance their cross-border payment system.",
    "Several cryptocurrency platforms are considering investing in Polygon to expand their payment system capabilities.",
    "The founder of Ripple revealed that the company Ripple has invested in several cryptocurrency platforms to facilitate the everyday use of digital currencies.",
    "Ripple joined forces with Visa to enhance and upgrade its payment system.",
    "Cardano and Ethereum partnered to develop a blockchain-based platform for the gaming industry.",
    "Stellar and IBM partnered to develop a blockchain-based supply chain management platform.",
    "Ripple partnered Stellar to enable users to send money internationally.",
    "Ripple is exploring investment in Polygon to create a more integrated payment ecosystem that leverages the strengths of both cryptocurrencies to provide its users with a more seamless payment experience.",
    "Visa partners with Cardano to enable customers to use digital currencies for everyday purchases, making it easier and more convenient to transact with cryptocurrencies.",
    "Ripple has forged partnership with banks, including Banco Santander, Bank of America, and Standard Chartered, to facilitate cross-border payments using its blockchain technology and digital assets.",
    "banks are investing in Cardano's technology to improve their payment system capabilities.",
    "IBM invested in Cardano and caused a surge in interest and investment in the cryptocurrency market.",
    "Tesla invested in Etherium and announced plans to accept it as a form of payment for their electric vehicles, which caused a significant increase in Ethereum's value and attracted further investment in the cryptocurrency.",
    "Visa invested in Polygon and is exploring ways to incorporate its technology into their payment system to provide faster and more efficient transactions, which has attracted further investment in Polygon and its potential for growth in the cryptocurrency market.",
    "IBM invested in Cardano recognizing the potential for growth and development in the cryptocurrency industry.",
    "Tesla invested in BTC and made headlines when they announced that they had allocated a portion of their treasury reserves to Bitcoin, signaling a growing acceptance of cryptocurrency as a legitimate investment option",
    "Visa is seeking for software analyst to develop cryptocurrency platforms partnered with Cardano",
    "IBM is hiring for a software analyst based on expertise in Cardano",
    "IBM saw adaptation of Ethereum profitable based on the return on investment",
    "The founder of Ethereum, Vitalik Buterin stated in 2023 that they will buy Ripple based on its development",
    "Ripple in partnership with Polygon to improve their payment system is another exciting development in the rapidly evolving cryptocurrency market, driven by the innovative thinking of developers like Vitalik Buterin.",
    "Vitalik Buterin invested in Ethereum in its early stages of development, and played a crucial role in its creation and growth as one of the most widely used and innovative blockchain platforms in the world."
]

In [ ]:
custom_nlp = spacy.load("./drive/MyDrive/ie_course_2022_team08/output/ml_custom_ner/model-last")
nlp = spacy.load("en_core_web_lg")
custom_nlp.replace_listeners("tok2vec", "ner", ["model.tok2vec"])
nlp.add_pipe("ner", source=custom_nlp, name="custom_ner", before="ner")

# create dependency and phrase matchers
dep_matcher = DependencyMatcher(nlp.vocab, validate=True)
rel_matcher = PhraseMatcher(nlp.vocab, None)
ent_matcher = PhraseMatcher(nlp.vocab, None)

# create pattern of phrase matcher for relations using gazetteers
rel_patterns = [nlp.make_doc(gazetteer) for gazetteer in relations_dictionary]
rel_matcher.add("relations", rel_patterns)

# create pattern of phrase matcher for entities using gazetteers
ent_patterns = [nlp.make_doc(gazetteer) for gazetteer in entities_dictionary]
ent_matcher.add("entities", ent_patterns)

In [ ]:
# SPO triple function
def extract_triples_from_sentence(sentence):

  # initial process of the sentence with the NLP pipeline
  doc = nlp(sentence)
  # display dependency visualization
  displacy.render(doc, style="dep", jupyter=True, options={"distance": 90})

  #################################################
  # 1. use phrase matcher to identify occurrences of predicates and entities
  #################################################

  # matches for relation (1 or many extractions)
  rel_matches = rel_matcher(doc)
  # get relation spans from relation matches
  rel_spans = [doc[start:end] for _, start, end in rel_matches]
  # filter overlaping spans to ensure uniqueness of relations
  relation_spans = spacy.util.filter_spans(rel_spans)
  # create list of extracted relations in string type
  relations = [r.text for r in relation_spans]

  # matches for entitties (1 or many extractions)
  ent_matches = ent_matcher(doc)
  # get entity spans from entity matches
  ent_spans = [doc[start:end] for _, start, end in ent_matches]
  # filter overlapping spans to ensure uniqueness of entities
  entity_spans = spacy.util.filter_spans(ent_spans)
  # create list of extracted entities in string type (*)
  entities = [e.text for e in entity_spans]

  print(f"• Extracted {len(relations)} relation(s): {relations}")
  print(f"• Extracted {len(entities)} entity(ies): {entities}")
  print()

  #################################################
  # 2. use relations (predicates) occurrence in sentence to identify entities
  #################################################

  # container of extracted triples. even though it may be unlikely,
  # a single sentence may contain more than 1 triple
  triples = []

  #################################################
  # NOTE: we start by identifying the position of the relation in the sentence.
  # Notice that a relation is a span object (consisting of more than 1 token,
  # more than 1 word), so the first step is identifying the head of the
  # relation and then the subject and object entities that constitute
  # the triple.
  #################################################

  for rel_span in relation_spans:
    # divide span in tokens
    for token in rel_span:
      # convert ancestors to string, so we can find the head of the relation span
      ancestors = [a.text for a in list(token.ancestors)]
      # find head of the relation span
      if token.text in ancestors:
        continue
      else:
        relation_head = token.text

    # container of extracted single SPO triple
    triple= []

    #################################################
    # NOTE: the DEP matcher pattern for SPO triples, starts locating the
    # (head) of the relation by string match ("TEXT") and then it moves to the
    # subject and object using dependency labels ("DEP"). This pattern will
    # return 1 or more matches of triples extracted from a single sentence
    # since a (long) sentence may contain more than one reference to the same
    # relation with the same or different entities.
    # SPO triple = Subject-Predicate-Object triple
    #################################################
    pattern = [
      {
        "RIGHT_ID": "predicate",
        "RIGHT_ATTRS": {"TEXT": relation_head}, # 1. relation (predicate)
      },
      {
        "LEFT_ID": "predicate",
        "REL_OP": ";*",
        "RIGHT_ID": "subject",
        "RIGHT_ATTRS": {"DEP": {"IN":["nsubj", "csubj","nsubjpass"]}}, # 2. subject
      },
      {
        "LEFT_ID": "predicate",
        "REL_OP": ">>",
        "RIGHT_ID": "object",
        "RIGHT_ATTRS": {"DEP": {"IN":["dobj", "pobj", "poss"]}}, # 3. object
      }
    ]

    # add patern to extract SPO triples using the DEP matcher
    dep_matcher.add("semantic_triple", [pattern])
    # extract matches of SPO triples
    pso_matches = dep_matcher(doc)

    # pass the entities (strings) through the pipeline to tokenize them
    entities_docs = [nlp(e) for e in entities]

    #################################################
    # 3. identify object and subject
    #################################################

    #################################################
    # NOTE: in this sample of a returned match (within "matches"),
    # (4699203773119030710, [12, 5, 15]), the second element in the
    # tuple (the list) corresponds to [P, S, O] found by the pattern.
    # Pay attention to it because we use the position of S and O in this
    # function. In the pattern we used only the head of the relation
    # so the returned match contains only a 1-word match, therefore we use
    # it now to find the corresponding entity in "entities", now converted
    # to a doc object in "entities_docs". Use use the Word2vec algorithm to
    # find similarity
    #################################################
    def reduce_entities(idx):
      # comparison of all entities found by the phrase matcher with the
      # returned candidates matched entities, only one will remain
      highest_similarity = 0
      ent = None
      for pso_match in pso_matches:
        candidate = doc[pso_match[1][idx]]
        doc_candidate = nlp(candidate.text)

        for e in entities_docs:
          similarity = doc_candidate.similarity(e)
          if highest_similarity < similarity:
            highest_similarity = similarity
            ent = e.text
      return ent

    # get the full name of the subject from the "entities" list (*)
    subj = reduce_entities(idx=1)
    # add subject to triple (S--)

    # add relation (predicate) to triple (-P-)
    # NOTE: we add the relation (predicate) between S and O
    rel = rel_span.text

    # get the full name of the object from the "entities" list (*)
    obj = reduce_entities(idx=2)

    # make sure the triple is well-formed to add Wikidata IDS
    # otherwise it doesn't update the container "triples" and returns empty "[]"
    if subj and rel and obj:
      def look_up_wikidata_id(component_type, label):
        wikidata_id = None
        if component_type == "entity":
          dict_to_look_up = entities_dicts
          id_type = "qid" # Q is for entities in Wikidata information model
        elif component_type == "relation":
          dict_to_look_up = relations_dicts
          id_type = "pid" # P is for proerties (relations) in Wikidata information model
        for d in dict_to_look_up:
          if d.get("aliases"): # if relation/entity has aliasses attrib
            if label == d["label"] or label in d["aliases"]:
              wikidata_id = d[id_type]
          else:
            if label == d["label"]:
              wikidata_id = d[id_type]
        return wikidata_id

      triple = [
          (look_up_wikidata_id("entity", subj), subj),
          (look_up_wikidata_id("relation", rel), rel),
          (look_up_wikidata_id("entity", obj), obj),
      ]
      triples.append(triple)

    # NOTE: pending task, ensure uniqueness of triples coming from the same
    # sentence, even though unlikely, it may happen

  return triples

In [ ]:
# extract SPO triples from sentences
triples_superlist = []
for sentence in sentences:
  triples_in_sentence = extract_triples_from_sentence(sentence)
  if len(triples_in_sentence):
    # print("Extracted SPO triples:")
    for t in triples_in_sentence:
      print(f"• {t}")
      triples_superlist.append(t)
      print()
  else:
    print("Not extracted SPO triples!")
    print()

  # NOTE: pending task, ensure uniqueness of triples coming
  # from the same text

• Extracted 1 relation(s): ['developed by']
• Extracted 2 entity(ies): ['Ethereum', 'Vitalik Buterin']

• [('Q29583124', 'Ethereum'), ('P61', 'developed by'), ('Q16197959', 'Vitalik Buterin')]



• Extracted 1 relation(s): ['investing']
• Extracted 2 entity(ies): ['Santander Bank', 'Polygon']

• [('Q5835668', 'Santander Bank'), ('P1951', 'investing'), ('Q40186999', 'Polygon')]



• Extracted 1 relation(s): ['partnership']
• Extracted 2 entity(ies): ['Polygon', 'Santander Bank']

• [('Q40186999', 'Polygon'), ('P1327', 'partnership'), ('Q5835668', 'Santander Bank')]



• Extracted 1 relation(s): ['developed by']
• Extracted 2 entity(ies): ['Ethereum Classic', 'Charlie Shrem']

• [('Q29583124', 'Ethereum Classic'), ('P61', 'developed by'), ('Q15711672', 'Charlie Shrem')]



• Extracted 1 relation(s): ['hiring']
• Extracted 2 entity(ies): ['Polygon', 'VC analyst']

• [('Q40186999', 'Polygon'), ('P899277', 'hiring'), ('Q485178', 'VC analyst')]



...

• Extracted 2 relation(s): ['hiring', 'based on']
• Extracted 3 entity(ies): ['IBM', 'analyst', 'Cardano']

• [('Q41273538', 'IBM'), ('P899277', 'hiring'), ('Q485178', 'analyst')]

• [('Q41273538', 'IBM'), ('P144', 'based on'), ('Q485178', 'analyst')]



• Extracted 2 relation(s): ['based on', 'investment']
• Extracted 2 entity(ies): ['IBM', 'Ethereum']

• [('Q41273538', 'IBM'), ('P144', 'based on'), ('Q29583124', 'Ethereum')]

• [('Q41273538', 'IBM'), ('P1951', 'investment'), ('Q29583124', 'Ethereum')]



• Extracted 1 relation(s): ['based on']
• Extracted 3 entity(ies): ['Ethereum', 'Vitalik Buterin', 'Ripple']

• [('Q16197959', 'Vitalik Buterin'), ('P144', 'based on'), ('Q29583124', 'Ethereum')]



• Extracted 1 relation(s): ['partnership']
• Extracted 4 entity(ies): ['Ripple', 'Polygon', 'cryptocurrency', 'Vitalik Buterin']

• [('Q1307473', 'Ripple'), ('P1327', 'partnership'), ('Q40186999', 'Polygon')]



• Extracted 1 relation(s): ['invested in']
• Extracted 2 entity(ies): ['Vitalik Buterin', 'Ethereum']

• [('Q16197959', 'Vitalik Buterin'), ('P1951', 'invested in'), ('Q29583124', 'Ethereum')]



Converting the triples to a form suitable for input to neo4j

In [ ]:
triples_GraphCompatible = []
for triple in triples_superlist:
    subj = triple[0][1]
    pred = triple[1][1]
    obj = triple[2][1]
    triples_GraphCompatible.append((subj, pred, obj))
print(triples_GraphCompatible)


[('Ethereum', 'developed by', 'Vitalik Buterin'), ('Santander Bank', 'investing', 'Polygon'), ('Polygon', 'partnership', 'Santander Bank'), ('Ethereum Classic', 'developed by', 'Charlie Shrem'), ('Polygon', 'hiring', 'VC analyst'), ('Ethereum Classic', 'hiring', 'Software analyst'), ('Charlie Shrem', 'discovered', 'Ethereum Classic'), ('Charlie Shrem', 'discovered', 'Ethereum Classic'), ('BTC', 'based on', 'BTC'), ('Ripple', 'partnered with', 'Visa'), ('banks', 'partners with', 'banks'), ('cryptocurrency platforms', 'partners with', 'cryptocurrency platforms'), ('Blockchain', 'based on', 'cryptocurrency'), ('Ripple', 'partnered with', 'Santander Bank'), ('Ripple', 'partnered with', 'Polygon'), ('Charlie Shrem', 'seeking', 'software analysts'), ('VC analyst', 'vacancy', 'VC analyst'), ('Visa', 'collaborates with', 'cryptocurrency platforms'), ('Ethereum', 'based on', 'Ripple'), ('IBM', 'invested in', 'Stellar'), ('IBM', 'investment', 'Stellar'), ('Santander Bank', 'joined forces', 'Visa

In [ ]:
# Only Unique tripes
unique_triples=list(set(triples_GraphCompatible))
print(unique_triples)
print(len(unique_triples))

[('IBM', 'invested in', 'Stellar'), ('Ripple', 'partnership', 'banks'), ('BTC', 'based on', 'BTC'), ('Visa', 'investment', 'Polygon'), ('Santander Bank', 'joined forces', 'Visa'), ('Cardano', 'partnered', 'Ethereum'), ('Blockchain', 'based on', 'cryptocurrency'), ('Tesla', 'investment', 'BTC'), ('Ripple', 'partnered with', 'Polygon'), ('Visa', 'partnered with', 'Cardano'), ('Visa', 'seeking', 'Cardano'), ('Tesla', 'invested in', 'Ethereum'), ('Stellar', 'partnered', 'IBM'), ('Ripple', 'invested in', 'cryptocurrency platforms'), ('banks', 'investing', 'Cardano'), ('Santander Bank', 'investing', 'Polygon'), ('cryptocurrency platforms', 'investing', 'Polygon'), ('Charlie Shrem', 'discovered', 'Ethereum Classic'), ('IBM', 'investment', 'Cardano'), ('Ripple', 'joined forces', 'Visa'), ('IBM', 'hiring', 'analyst'), ('Visa', 'invested in', 'Polygon'), ('Ripple', 'investment', 'Polygon'), ('Ethereum', 'developed by', 'Vitalik Buterin'), ('IBM', 'investment', 'Stellar'), ('Vitalik Buterin', 'ba

# Neo4J Code

In [ ]:
def get_obj_properties(tup_ls):

    init_obj_tup_ls = []

    for tup in tup_ls:

        try:
            text, node_label_ls, url = query_google(tup[2], api_key, limit=1)
            new_tup = (tup[0], tup[1], tup[2], text[0], node_label_ls[0], url[0])
        except:
            new_tup = (tup[0], tup[1], tup[2], [], [], [])

        init_obj_tup_ls.append(new_tup)

    return init_obj_tup_ls


def add_layer(tup_ls):

    svo_tup_ls = []

    for tup in tup_ls:

        if tup[3]:
            svo_tup = create_svo_triples(tup[3])
            svo_tup_ls.extend(svo_tup)
        else:
            continue

    return get_obj_properties(svo_tup_ls)

def subj_equals_obj(tup_ls):

    new_tup_ls = []

    for tup in tup_ls:
        if tup[0] != tup[2]:
            new_tup_ls.append((tup[0], tup[1], tup[2], tup[3], tup[4], tup[5]))

    return new_tup_ls


def check_for_string_labels(tup_ls):

    clean_tup_ls = []

    for el in tup_ls:
        if isinstance(el[2], list):
            clean_tup_ls.append(el)

    return clean_tup_ls


def create_word_vectors(tup_ls):

    new_tup_ls = []

    for tup in tup_ls:
        if tup[3]:
            doc = nlp(tup[3])
            new_tup = (tup[0], tup[1], tup[2], tup[3], tup[4], tup[5], doc.vector)
        else:
            new_tup = (tup[0], tup[1], tup[2], tup[3], tup[4], tup[5], np.random.uniform(low=-1.0, high=1.0, size=(300,)))
        new_tup_ls.append(new_tup)

    return new_tup_ls

In [ ]:
init_obj_tup_ls = get_obj_properties(unique_triples)
new_layer_ls = add_layer(init_obj_tup_ls)
starter_edge_ls = init_obj_tup_ls + new_layer_ls
edge_ls = subj_equals_obj(starter_edge_ls)
clean_edge_ls = edge_ls

In [ ]:
edges_word_vec_ls = create_word_vectors(edge_ls)

In [ ]:
def dedup(tup_ls):

    visited = set()
    output_ls = []

    for tup in tup_ls:
        if not tup[0] in visited:
            visited.add(tup[0])
            output_ls.append((tup[0], tup[1], tup[2], tup[3], tup[4]))

    return output_ls


def convert_vec_to_ls(tup_ls):

    vec_to_ls_tup = []

    for el in tup_ls:
        vec_ls = [float(v) for v in el[4]]
        tup = (el[0], el[1], el[2], el[3], vec_ls)
        vec_to_ls_tup.append(tup)

    return vec_to_ls_tup


def add_nodes(tup_ls):

    keys = ['name', 'description', 'node_labels', 'url', 'word_vec']
    merge_nodes(graph.auto(), tup_ls, ('Node', 'name'), keys=keys)
    print('Number of nodes in graph: ', graph.nodes.match('Node').count())

    return

In [ ]:
def add_edges(edge_ls):

    edge_dc = {}

    for tup in edge_ls:
        if tup[1] in edge_dc:
            edge_dc[tup[1]].append((tup[0], tup[1], tup[2]))
        else:
            edge_dc[tup[1]] = [(tup[0], tup[1], tup[2])]

    for edge_labels, tup_ls in tqdm(edge_dc.items()):   # k=edge labels, v = list of tuples

        tx = graph.begin()

        for el in tup_ls:
            source_node = nodes_matcher.match(name=el[0]).first()
            target_node = nodes_matcher.match(name=el[2]).first()
            if not source_node:
                source_node = Node('Node', name=el[0])
                tx.create(source_node)
            if not target_node:
                try:
                    target_node = Node('Node', name=el[2], node_labels=el[4], url=el[5], word_vec=el[6])
                    tx.create(target_node)
                except:
                    continue
            try:
                rel = Relationship(source_node, edge_labels, target_node)
            except:
                continue
            tx.create(rel)
        tx.commit()

    return

In [ ]:
orig_node_tup_ls = [(edge_ls[0][0], '', ['Subject'], '', np.random.uniform(low=-1.0, high=1.0, size=(300,)))]
obj_node_tup_ls = [(tup[2], tup[3], tup[4], tup[5], tup[6]) for tup in edges_word_vec_ls]
full_node_tup_ls = orig_node_tup_ls + obj_node_tup_ls
dedup_node_tup_ls = dedup(full_node_tup_ls)

In [ ]:
len(full_node_tup_ls), len(dedup_node_tup_ls)

(47, 20)

In [ ]:
node_tup_ls = convert_vec_to_ls(dedup_node_tup_ls)

In [ ]:
# open Neo4j credentials
uri = neo4j_credential["uri"]
user = neo4j_credential["user"]
pwd = neo4j_credential["password"]

graph = Graph(uri, auth=('neo4j', pwd))
nodes_matcher = NodeMatcher(graph)

In [ ]:
add_nodes(node_tup_ls)

Number of nodes in graph:  26


In [ ]:
add_edges(edges_word_vec_ls)

  0%|          | 0/15 [00:00<?, ?it/s]<ipython-input-201-feb92114d42c>:36: DeprecationWarning: The transaction.commit() method is deprecated, use graph.commit(transaction) instead
  tx.commit()
100%|██████████| 15/15 [00:16<00:00,  1.08s/it]
